# Notebook 9: Reviewer checks (near-duplicates and job-clustered intervals)

**Runtime:** CPU is enough (no GPU needed). Run time about 10 minutes.

**Why this notebook:** Mr. Shohel Arman's review (19 Oct) asked for two checks the paper does not have yet.

**Part A. Near-duplicate resumes and jobs.** Our resume and job IDs come from a hash of the *exact* cleaned text. Two resumes that differ only slightly therefore get different IDs, so the "strict" split could still contain a near-copy of a test resume in training. We find near-duplicates, count how many test pairs they touch in the standard/strict split and in the 5×5 grid of Notebook 6a, and then recompute the 6a results **without** the affected test pairs, using the scores 6a already saved (no retraining).

**Part B. Job-clustered bootstrap for the human labels.** Notebook 5 and the second-annotator analysis resampled single *pairs*. Pairs from the same job are not independent (150 pairs come from only 42 jobs), so we redo the confidence intervals by resampling whole **jobs**. Point estimates do not change; only the intervals and p-values can.

**Rules fixed before running (also in ANALYSIS-PLAN.md, addendum 2026-10-19):**
- A **near-duplicate** is a pair of different texts with TF-IDF cosine ≥ 0.90 **or** word 5-shingle Jaccard ≥ 0.50. Thresholds 0.80 and 0.95 are reported as sensitivity only.
- A grid test pair is **affected** if its resume has a near-duplicate among that fold's *strict* training resumes, or its job has a near-duplicate among that fold's training jobs.
- If affected pairs are under 2% of the test pairs, the paper says near-duplicates are rare and do not explain the result. Otherwise the paper reports the clean-subset numbers. The memorisation conclusion **holds** if *size-matched − strict* keeps its sign with a CI that excludes 0 on the clean subset, for both AUC fit-vs-no and NDCG@5.
- Part B: if a CI that excluded 0 includes it under job clustering (or the reverse), the paper says so in Section 6.3.

**Needs on Drive:** `data/train|val|test.parquet`, `results/cv_cache/` (Notebook 6a), `results/05_labelled_pairs.csv` (Notebook 5). Writes to `results/09_review_checks/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, warnings
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from scipy.stats import wilcoxon
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, cohen_kappa_score
warnings.filterwarnings('ignore', category=RuntimeWarning)

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/cv_cache'
OUT = f'{RESULTS}/09_review_checks'
os.makedirs(OUT, exist_ok=True)
SEED = 42; K = 5
COS, JAC = 0.90, 0.50          # near-duplicate rule, fixed in advance

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
all_df['pair_key'] = all_df.job_id + '|' + all_df.resume_id
res_text = all_df.drop_duplicates('resume_id').set_index('resume_id').resume_text
job_text = all_df.drop_duplicates('job_id').set_index('job_id').job_text
print(f'{len(all_df)} pairs, {len(job_text)} jobs, {len(res_text)} resumes')
assert len(job_text) == 351 and len(res_text) == 643, 'not the same data as the paper'

## A1. Find near-duplicate texts
Two measures, both on the cleaned text:
- **TF-IDF cosine**, with the same TF-IDF settings as the paper (fitted on all job and resume texts, no labels). It catches texts with the same words.
- **Word 5-shingle Jaccard**: the share of 5-word sequences the two texts have in common. It catches copied passages.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(job_text) + list(res_text))

def words(t):
    return re.sub(r'[^a-z0-9]+', ' ', t.lower()).split()

def shingles(t, n=5):
    w = words(t)
    return {' '.join(w[i:i + n]) for i in range(max(1, len(w) - n + 1))}

def similar_pairs(texts):
    ids = list(texts.index)
    X = tfidf.transform(texts.tolist())
    cos = (X @ X.T).toarray()
    sh = [shingles(t) for t in texts]
    rows = []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            inter = len(sh[i] & sh[j])
            jac = inter / (len(sh[i]) + len(sh[j]) - inter) if inter else 0.0
            if cos[i, j] >= 0.5 or jac >= 0.2:
                rows.append((ids[i], ids[j], cos[i, j], jac))
    np.fill_diagonal(cos, -1)
    nn = pd.Series(cos.max(axis=1), index=ids)          # similarity to the closest OTHER text
    return pd.DataFrame(rows, columns=['a', 'b', 'cosine', 'jaccard']), nn

def near_groups(pairs, ids, cos=COS, jac=JAC):
    p = pairs[(pairs.cosine >= cos) | (pairs.jaccard >= jac)]
    pos = {x: i for i, x in enumerate(ids)}
    g = sparse.coo_matrix((np.ones(len(p)), (p.a.map(pos), p.b.map(pos))), shape=(len(ids), len(ids)))
    _, comp = connected_components(g, directed=False)
    neigh = {x: set() for x in ids}
    for a, b in zip(p.a, p.b):
        neigh[a].add(b); neigh[b].add(a)
    return p, pd.Series(comp, index=ids), neigh

res_pairs, res_nn = similar_pairs(res_text)
job_pairs, job_nn = similar_pairs(job_text)

summary_a1 = []
for kind, pairs, nn, ids in [('resumes', res_pairs, res_nn, list(res_text.index)), ('jobs', job_pairs, job_nn, list(job_text.index))]:
    for c, j in [(0.80, 0.30), (COS, JAC), (0.95, 0.70)]:
        p, _, neigh = near_groups(pairs, ids, c, j)
        summary_a1.append({'texts': kind, 'rule': f'cos>={c} or jac>={j}' + ('  (MAIN)' if c == COS else ''),
                           'near-duplicate pairs': len(p), 'texts with a near-duplicate': sum(len(v) > 0 for v in neigh.values()),
                           'of': len(ids)})
    print(f'{kind}: closest other text, TF-IDF cosine quantiles 50/90/99/max =',
          np.round(np.percentile(nn, [50, 90, 99, 100]), 3))
summary_a1 = pd.DataFrame(summary_a1)
display(summary_a1)

res_near, res_group, res_neigh = near_groups(res_pairs, list(res_text.index))
job_near, job_group, job_neigh = near_groups(job_pairs, list(job_text.index))

### A1b. Look at the closest pairs
The five most similar resume pairs and job pairs (first 150 characters of each). Check by eye: are these really the same resume or job with small edits, or just two similar documents (for example two resumes built from the same template)? Write one line about what you see; it goes in the paper.

In [ ]:
def show(pairs, texts, n=5):
    top = pairs.sort_values(['cosine', 'jaccard'], ascending=False).head(n)
    for _, r in top.iterrows():
        print(f"cosine {r.cosine:.3f}  jaccard {r.jaccard:.3f}")
        print('   A:', ' '.join(texts[r.a].split())[:150])
        print('   B:', ' '.join(texts[r.b].split())[:150])
        print()
print('=== RESUMES ==='); show(res_pairs, res_text)
print('=== JOBS ==='); show(job_pairs, job_text)

## A2. The standard and strict split (Notebooks 3 and 4)
Exact overlap first (this must reproduce the paper's 430 of 434), then near-duplicates that survive the strict split.

In [ ]:
test = all_df[all_df.split == 'test']; train = all_df[all_df.split == 'train']
test_r, train_r = set(test.resume_id), set(train.resume_id)
print(f'exact: {len(test_r & train_r)} of {len(test_r)} test resumes also appear in training (paper: 430 of 434)')

strict_train = train[~train.resume_id.isin(test_r)]
strict_r = set(strict_train.resume_id); train_j = set(train.job_id)
leak_r = {r for r in test_r if res_neigh[r] & strict_r}
leak_j = {j for j in set(test.job_id) if job_neigh[j] & train_j}
aff = test.resume_id.isin(leak_r) | test.job_id.isin(leak_j)
split_tab = pd.DataFrame([
    {'check': 'test resumes with a near-duplicate in STRICT training', 'count': len(leak_r), 'of': len(test_r)},
    {'check': 'test jobs with a near-duplicate job in training', 'count': len(leak_j), 'of': test.job_id.nunique()},
    {'check': 'test pairs affected (either)', 'count': int(aff.sum()), 'of': len(test)},
])
display(split_tab)

## A3. The 5×5 grid of Notebook 6a, without the affected test pairs
Same folds as 6a (same seed and code). First the 6a numbers are **reproduced on all test pairs** (the check must say OK), then recomputed on the clean subset.

In [ ]:
rng = np.random.default_rng(SEED)
jobs = all_df.job_id.unique(); res = all_df.resume_id.unique()
job_fold = dict(zip(jobs, rng.permutation(np.arange(len(jobs)) % K)))
res_fold = dict(zip(res, rng.permutation(np.arange(len(res)) % K)))
all_df['jf'] = all_df.job_id.map(job_fold); all_df['rf'] = all_df.resume_id.map(res_fold)

def fold_test(k):
    return all_df[(all_df.jf == k) & (all_df.rf == k)]

flags = []
for k in range(K):
    t = fold_test(k)
    strict_res = set(all_df[(all_df.jf != k) & (all_df.rf != k)].resume_id)
    train_jobs = set(all_df[all_df.jf != k].job_id)
    flags.append(pd.DataFrame({'pair_key': t.pair_key,
                               'resume_leak': [bool(res_neigh[r] & strict_res) for r in t.resume_id],
                               'job_leak': [bool(job_neigh[j] & train_jobs) for j in t.job_id]}))
flags = pd.concat(flags, ignore_index=True)
flags['affected'] = flags.resume_leak | flags.job_leak

test_all = pd.concat([fold_test(k).assign(fold=k) for k in range(K)], ignore_index=True)
ref = pd.read_parquet(f'{CACHE}/reference.parquet')
ev = test_all[['pair_key', 'job_id', 'resume_id', 'fold', 'relevance', 'label']].merge(ref, on='pair_key').reset_index(drop=True)
for name in ['standard', 'strict', 'size-matched']:
    s = pd.concat([pd.read_parquet(f'{CACHE}/fold{k}_{name}.parquet') for k in range(K)])
    ev = ev.merge(s.rename(columns={'score': f'FT {name}', 'prior': f'Resume-prior ({name} train)'}), on='pair_key').reset_index(drop=True)
ev = ev.merge(flags, on='pair_key')
share = ev.affected.mean()
print(f'{len(ev)} grid test pairs; affected: {int(ev.affected.sum())} ({share:.1%}) '
      f'[resume {int(ev.resume_leak.sum())}, job {int(ev.job_leak.sum())}]')
print('RULE: under 2% -> "rare, does not explain the result"' if share < 0.02 else 'RULE: 2% or more -> report the clean-subset numbers')

In [ ]:
# metric code copied from Notebook 6a so the numbers reproduce exactly
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k]); return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def auc(y, s):
    y = np.asarray(y); return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan
def pooled(df, col):
    fit = df[df.relevance > 0]
    return {'AUC fit-vs-no': auc((df.relevance > 0).astype(int), df[col]),
            'AUC good-vs-potential': auc((fit.relevance == 2).astype(int), fit[col])}
def per_job(df, col):
    r = np.random.default_rng(SEED); rows = []
    for jid, g in df.groupby('job_id'):
        rel = g.relevance.to_numpy(); s = g[col].to_numpy(); fit = rel > 0
        row = {'job_id': jid}
        if len(g) >= 3 and len(set(rel)) >= 2:
            row['NDCG@5'] = ndcg_at_k(rel[np.lexsort((r.random(len(s)), -s))].tolist(), 5)
        row['job AUC fit-vs-no'] = auc(fit.astype(int), s)
        row['job AUC good-vs-potential'] = auc((rel[fit] == 2).astype(int), s[fit]) if fit.sum() >= 2 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('job_id')

METHODS = ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'E5-base (zero-shot)', 'TF-IDF']
def summarise(d, tag):
    rows = []
    for m in METHODS:
        p = pooled(d, m); pj = per_job(d, m)
        rows.append({'subset': tag, 'method': m, 'pairs': len(d), 'jobs': d.job_id.nunique(),
                     'AUC fit-vs-no': round(p['AUC fit-vs-no'], 3), 'AUC good-vs-potential': round(p['AUC good-vs-potential'], 3),
                     'NDCG@5': round(pj['NDCG@5'].mean(), 3), 'NDCG@5 jobs': int(pj['NDCG@5'].notna().sum())})
    return pd.DataFrame(rows)

full = summarise(ev, 'all test pairs')
PAPER = {('FT standard', 'AUC fit-vs-no'): 0.795, ('FT size-matched', 'AUC fit-vs-no'): 0.787,
         ('FT strict', 'AUC fit-vs-no'): 0.758, ('FT standard', 'NDCG@5'): 0.857, ('FT strict', 'NDCG@5'): 0.781}
bad = [(m, c, v, full.set_index('method').at[m, c]) for (m, c), v in PAPER.items()
       if abs(full.set_index('method').at[m, c] - v) > 0.0015]
print('Reproduction of 6a:', 'OK' if not bad else f'MISMATCH {bad}  <- stop and')

clean = ev[~ev.affected].reset_index(drop=True)
grid_tab = pd.concat([full, summarise(clean, 'without affected pairs')], ignore_index=True) if len(clean) < len(ev) else full
display(grid_tab)

### A3b. Paired differences on the clean subset
The same tests as 6a/6c: AUC differences with a bootstrap over test jobs (2,000 resamples), and NDCG@5 per-job differences with a job bootstrap and a Wilcoxon signed-rank test. Positive = the first model is better.

In [ ]:
def paired(d, a, b, n=2000):
    r = np.random.default_rng(SEED); idx = list(d.groupby('job_id').indices.values()); out = []
    for metric in ['AUC fit-vs-no', 'AUC good-vs-potential']:
        f = lambda x: pooled(x, a)[metric] - pooled(x, b)[metric]
        base = f(d); boots = np.array([f(d.iloc[np.concatenate([idx[i] for i in r.integers(0, len(idx), len(idx))])]) for _ in range(n)])
        boots = boots[~np.isnan(boots)]; lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append({'comparison': f'{a} - {b}', 'metric': metric, 'diff': round(base, 3), 'CI': f'[{lo:.3f}, {hi:.3f}]',
                    'p': round(min(1, 2 * min((boots <= 0).mean(), (boots >= 0).mean())), 4)})
    da = per_job(d, a)['NDCG@5']; db = per_job(d, b)['NDCG@5']; diff = (da - db).dropna().to_numpy()
    boots = np.array([diff[r.integers(0, len(diff), len(diff))].mean() for _ in range(n)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    out.append({'comparison': f'{a} - {b}', 'metric': 'NDCG@5', 'diff': round(diff.mean(), 3), 'CI': f'[{lo:.3f}, {hi:.3f}]',
                'p': round(wilcoxon(diff).pvalue, 4) if np.any(diff != 0) else 1.0})
    return out

PAIRS = [('FT standard', 'FT strict'), ('FT size-matched', 'FT strict'), ('FT standard', 'FT size-matched'),
         ('FT strict', 'BGE-base (zero-shot)'), ('FT strict', 'TF-IDF')]
subsets = [('all test pairs', ev)] + ([('without affected pairs', clean)] if len(clean) < len(ev) else [])
tests_a = pd.DataFrame([dict(subset=tag, **row) for tag, d in subsets for a, b in PAIRS for row in paired(d, a, b)])
display(tests_a)

if len(clean) < len(ev):
    t = tests_a[(tests_a.subset == 'without affected pairs') & (tests_a.comparison == 'FT size-matched - FT strict')]
    ok = all((r['diff'] > 0) and (float(r['CI'].strip('[]').split(',')[0]) > 0) for _, r in t[t.metric.isin(['AUC fit-vs-no', 'NDCG@5'])].iterrows())
    print('RULE: memorisation conclusion', 'HOLDS on the clean subset' if ok else 'does NOT hold on the clean subset ->')
else:
    print('No affected pairs: the clean subset is the full test set, nothing to recompute.')

## B. Job-clustered bootstrap for the human labels
**B1** reproduces Notebook 5's κ values and pair-bootstrap intervals, then adds intervals from resampling **whole jobs** (2,000 resamples). **B2** does the same for the Good-vs-Potential AUC shifts of Table 6 in the paper. **B3** does the second annotator's 40 pairs.

In [ ]:
lab = pd.read_csv(f'{RESULTS}/05_labelled_pairs.csv')
assert len(lab) == 150 and lab.h.notna().all(), 'expected the FINAL 150-pair file from Notebook 5'
print(f'{len(lab)} pairs from {lab.job_id.nunique()} jobs; pairs per job: median {lab.groupby("job_id").size().median():.0f}, '
      f'max {lab.groupby("job_id").size().max()}')

# second annotator, round 2 (labels only; copied from data/second_labeller/round2.xlsx in the repo)
H2 = {4: 'Good Fit', 7: 'No Fit', 8: 'Potential Fit', 18: 'Potential Fit', 23: 'No Fit', 26: 'No Fit', 29: 'Potential Fit',
      30: 'No Fit', 31: 'Potential Fit', 46: 'No Fit', 54: 'No Fit', 55: 'Potential Fit', 58: 'Good Fit', 63: 'No Fit',
      66: 'Good Fit', 68: 'Potential Fit', 70: 'Good Fit', 79: 'Potential Fit', 80: 'No Fit', 85: 'Potential Fit',
      89: 'Potential Fit', 95: 'Potential Fit', 97: 'Potential Fit', 98: 'No Fit', 101: 'Good Fit', 102: 'Potential Fit',
      104: 'Potential Fit', 107: 'Good Fit', 112: 'Potential Fit', 119: 'No Fit', 124: 'No Fit', 125: 'No Fit', 126: 'No Fit',
      134: 'Potential Fit', 136: 'No Fit', 137: 'Good Fit', 139: 'No Fit', 141: 'No Fit', 144: 'No Fit', 148: 'Good Fit'}
LEVEL = {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}
lab['h2'] = lab.pair_no.map(H2).map(LEVEL)
assert lab.h2.notna().sum() == 40

DECISIONS = {
    '3 labels':          lambda d, x, y: (d[x], d[y]),
    'fit vs no fit':     lambda d, x, y: ((d[x] > 0).astype(int), (d[y] > 0).astype(int)),
    'good vs potential': lambda d, x, y: (lambda f: ((f[x] == 2).astype(int), (f[y] == 2).astype(int)))(d[(d[x] > 0) & (d[y] > 0)]),
}
def kap(d, dec, x, y):
    a, b = DECISIONS[dec](d, x, y)
    return cohen_kappa_score(a, b) if len(a) and (len(set(a)) > 1 or len(set(b)) > 1) else np.nan

def kappa_ci_pairs(a, b, n=2000):            # copied from Notebook 5
    rng = np.random.default_rng(SEED); a, b = np.asarray(a), np.asarray(b); ks = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        if len(set(a[i])) > 1 or len(set(b[i])) > 1:
            ks.append(cohen_kappa_score(a[i], b[i]))
    return np.percentile(ks, [2.5, 97.5])

def job_resamples(d, n=2000):
    r = np.random.default_rng(SEED); idx = list(d.reset_index(drop=True).groupby('job_id').indices.values())
    d0 = d.reset_index(drop=True)
    return [d0.iloc[np.concatenate([idx[i] for i in r.integers(0, len(idx), len(idx))])] for _ in range(n)]

def fmt(lo, hi): return f'[{lo:.3f}, {hi:.3f}]'
def excl0(lo, hi): return 'yes' if lo > 0 or hi < 0 else 'no'

In [ ]:
# B1: author vs dataset, 150 pairs (Notebook 5 Table / paper Section 6.3)
B150 = job_resamples(lab)
rows = []
for dec in DECISIONS:
    a, b = DECISIONS[dec](lab, 'h', 'd')
    lo, hi = kappa_ci_pairs(a, b)
    clo, chi = np.nanpercentile([kap(s, dec, 'h', 'd') for s in B150], [2.5, 97.5])
    rows.append({'decision': dec, 'pairs': len(a), 'kappa': round(kap(lab, dec, 'h', 'd'), 3),
                 'CI pairs (NB5)': fmt(lo, hi), 'CI jobs (clustered)': fmt(clo, chi),
                 'excludes 0: pairs': excl0(lo, hi), 'excludes 0: jobs': excl0(clo, chi)})
b1 = pd.DataFrame(rows)
display(b1)
print('Paper values: 0.120 [0.015, 0.233], 0.324 [0.177, 0.476], -0.184 [-0.415, 0.063]')

In [ ]:
# B2: Good-vs-Potential AUC, human labels minus dataset labels (paper Table 6)
METH = [m for m in ['Resume-prior', 'TF-IDF', 'E5-base', 'BGE-base', 'MPNet-base', 'Qwen3-4B judge', 'Qwen3-1.7B judge'] if m in lab]
def gp(df, col, m):
    f = df[(df[col] > 0) & df[m].notna()]; y = (f[col] == 2).astype(int)
    return roc_auc_score(y, f[m]) if 0 < y.sum() < len(y) else np.nan

def shift_pairs(m, n=2000):                  # copied from Notebook 5
    rng = np.random.default_rng(SEED); diffs = []
    for _ in range(n):
        b = lab.iloc[rng.integers(0, len(lab), len(lab))]
        diffs.append(gp(b, 'h', m) - gp(b, 'd', m))
    return np.asarray(diffs)

def ci_p(diffs):
    diffs = diffs[~np.isnan(diffs)]; lo, hi = np.percentile(diffs, [2.5, 97.5])
    return lo, hi, min(1, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))

def holm(ps):
    ps = np.asarray(ps); o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
    for rank, i in enumerate(o):
        run = max(run, (len(ps) - rank) * ps[i]); adj[i] = min(1, run)
    return adj

rows = []
for m in METH:
    base = gp(lab, 'h', m) - gp(lab, 'd', m)
    lo, hi, p = ci_p(shift_pairs(m))
    clo, chi, cp = ci_p(np.array([gp(s, 'h', m) - gp(s, 'd', m) for s in B150]))
    rows.append({'method': m, 'shift': round(base, 3), 'CI pairs (NB5)': fmt(lo, hi), 'p pairs': round(p, 4),
                 'CI jobs': fmt(clo, chi), 'p jobs': round(cp, 4)})
b2 = pd.DataFrame(rows)
b2['Holm p jobs (job-aware methods)'] = np.nan
ja = b2.method != 'Resume-prior'
b2.loc[ja, 'Holm p jobs (job-aware methods)'] = np.round(holm(b2.loc[ja, 'p jobs']), 4)
display(b2)
print('Paper: Resume-prior -0.114 (p 0.211), TF-IDF +0.372, E5 +0.221 (p 0.011), BGE +0.195, MPNet +0.232, Qwen3-4B +0.359, Qwen3-1.7B +0.266')

In [ ]:
# B3: second annotator, 40 pairs (paper Section 6.3, "A second annotator")
l2 = lab[lab.h2.notna()].copy(); l2['h2'] = l2.h2.astype(int)
print(f'{len(l2)} pairs from {l2.job_id.nunique()} jobs')
B40 = job_resamples(l2)
r = np.random.default_rng(SEED); P40 = [l2.iloc[r.integers(0, len(l2), len(l2))] for _ in range(2000)]
rows = []
for dec in DECISIONS:
    for x, y, name in [('h2', 'h', 'H2-H1'), ('h2', 'd', 'H2-D'), ('h', 'd', 'H1-D')]:
        lo, hi = np.nanpercentile([kap(s, dec, x, y) for s in P40], [2.5, 97.5])
        clo, chi = np.nanpercentile([kap(s, dec, x, y) for s in B40], [2.5, 97.5])
        rows.append({'decision': dec, 'comparison': name, 'kappa': round(kap(l2, dec, x, y), 3),
                     'CI pairs': fmt(lo, hi), 'CI jobs': fmt(clo, chi)})
    est = kap(l2, dec, 'h2', 'h') - kap(l2, dec, 'h2', 'd')
    lo, hi = np.nanpercentile([kap(s, dec, 'h2', 'h') - kap(s, dec, 'h2', 'd') for s in P40], [2.5, 97.5])
    clo, chi = np.nanpercentile([kap(s, dec, 'h2', 'h') - kap(s, dec, 'h2', 'd') for s in B40], [2.5, 97.5])
    rows.append({'decision': dec, 'comparison': 'paired diff (H2-H1) - (H2-D)', 'kappa': round(est, 3),
                 'CI pairs': fmt(lo, hi), 'CI jobs': fmt(clo, chi)})
b3 = pd.DataFrame(rows)
display(b3)
print('Paper: H2-H1 0.115 (3 labels), 0.093 (fit/no); H2-D fit/no 0.319; paired diff fit/no -0.226 [-0.550, 0.119]')
print('Note: the paper CIs for these 40 pairs came from a JavaScript bootstrap, so "CI pairs" here is close but not identical.')

## Save and summary

In [ ]:
summary_a1.to_csv(f'{OUT}/A1_near_duplicates.csv', index=False)
pd.concat([res_near.assign(texts='resumes'), job_near.assign(texts='jobs')]).to_csv(f'{OUT}/A1_near_duplicate_pairs.csv', index=False)
split_tab.to_csv(f'{OUT}/A2_standard_strict_split.csv', index=False)
grid_tab.to_csv(f'{OUT}/A3_grid_clean.csv', index=False)
tests_a.to_csv(f'{OUT}/A3_grid_paired_tests.csv', index=False)
b1.to_csv(f'{OUT}/B1_kappa_clustered.csv', index=False)
b2.to_csv(f'{OUT}/B2_gp_shift_clustered.csv', index=False)
b3.to_csv(f'{OUT}/B3_second_annotator_clustered.csv', index=False)
print('saved to', OUT)
print()
print('SUMMARY')
print(f"  near-duplicate resume pairs: {len(res_near)}, job pairs: {len(job_near)} (cos>={COS} or jac>={JAC})")
print(f"  strict split: {len(leak_r)} test resumes have a near-copy in strict training; affected test pairs {int(aff.sum())}/{len(test)}")
print(f"  6a grid: affected test pairs {int(ev.affected.sum())}/{len(ev)} ({share:.1%})")
for _, rw in b1.iterrows():
    print(f"  kappa {rw['decision']}: {rw['kappa']}  pairs {rw['CI pairs (NB5)']}  jobs {rw['CI jobs (clustered)']}")